0.	p8s_pairs로 hidden state 재추출 + behavior 라벨 유도	L4 ~25–40분 / A100 ~10–15분
1.	w 추정 → cos(균형w, 원본w), diff-of-means 방향 확인	CPU ~5분
2.	steering 본실험: layer-block(front L12~13 / signal L15~20) × coef sweep × 랜덤대조	L4 ~70–90분 / A100 ~30–40분
3.	친구 실험 빈칸: cluster bootstrap CI / att_ans 토큰통제 / 행동축 재해석	CPU ~15분
4.	question-only vs context attention 대조 (교수님 미수행 지시)	L4 ~30–40분 / A100 ~12–18분

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import os, re, json, string, gc
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

MODEL_ID = "NousResearch/Meta-Llama-3.1-8B-Instruct"
SEED, MAX_NEW = 42, 48
OUT_DIR = "./p8_steer"; os.makedirs(OUT_DIR, exist_ok=True)

DRIVE_DIR = '/content/drive/MyDrive/Grad_share/data_share/데이터셋 모음'   # p8s_pairs.csv 위치
SAVE_DIR  = "/content/drive/MyDrive/Grad_share/data_share/P9"; os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
def normalize(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())

def match_gold(text, golds):
    t = normalize(text)
    for g in golds:
        if not str(g).strip(): continue
        if re.search(r"\b"+re.escape(normalize(g))+r"\b", t): return True
    return False

REFUSAL_PAT = re.compile(
    r"(i (don'?t|do not) know|couldn'?t find|cannot|unable to|no information|not sure)",
    re.IGNORECASE)

pairs = pd.read_csv(f"{DRIVE_DIR}/p8s_pairs.csv")
print(len(pairs), "행")
print(pairs.groupby(["kside","ctx_label"]).size().unstack(fill_value=0))
print("컬럼:", list(pairs.columns))

1224 행
ctx_label   C+   C-
kside              
K+         378  378
K−         234  234
컬럼: ['question', 'gold', 'prop', 'kside', 'knows', 'A_cb', 'ctx_id', 'ctx_title', 's_wiki_title', 'orig_gold', 'ctx_label', 'ctx_text', 'answer_surface', 'ans_start', 'ans_end', 'wrong_answer']


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()

N_LAYERS = model.config.num_hidden_layers   # 32
D_MODEL  = model.config.hidden_size         # 4096
print(f"로드 완료: {N_LAYERS} layers x {D_MODEL} dim")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

로드 완료: 32 layers x 4096 dim


In [ ]:
USER_TMPL = "{ctx}\n\nQuestion: {q}\nAnswer:"

def build_prompt(row):
    user = USER_TMPL.format(ctx=row.ctx_text, q=row.question)
    inputs = tok.apply_chat_template(
        [{"role": "user", "content": user}],
        add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True)
    return {k: v.to(model.device) for k, v in inputs.items()}

# 검증 1건
r0 = pairs.iloc[0]
inp = build_prompt(r0)
print("프롬프트 토큰 수:", inp["input_ids"].shape[1])
print("마지막 토큰:", repr(tok.decode(inp["input_ids"][0, -1])))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


프롬프트 토큰 수: 201
마지막 토큰: '\n\n'


In [ ]:
def build_prompt_str(row):
    user = USER_TMPL.format(ctx=row.ctx_text, q=row.question)
    return tok.apply_chat_template(
        [{"role":"user","content":user}],
        add_generation_prompt=True, tokenize=False)

r0 = pairs.iloc[0]
a = build_prompt(r0)["input_ids"].shape[1]                          # CELL3 방식
b = tok(build_prompt_str(r0), return_tensors="pt",
        add_special_tokens=False)["input_ids"].shape[1]             # CELL4 방식
print(a, b, "일치" if a==b else "★불일치 — CELL4 수정 필요")

201 201 일치


In [ ]:
r0 = pairs.iloc[0]
a = build_prompt(r0)["input_ids"].shape[1]                          # CELL3 방식 (tokenize=True)
b = tok(build_prompt_str(r0), return_tensors="pt",
        add_special_tokens=False)["input_ids"].shape[1]             # CELL4 방식
print(a, b, "일치" if a==b else "★불일치 — CELL4 수정 필요")

201 201 일치


In [ ]:
@torch.no_grad()
def gen_and_hidden(row):
    full = build_prompt_str(row)
    enc = tok(full, return_tensors="pt", add_special_tokens=False).to(model.device)
    L = enc["input_ids"].shape[1]
    # 1) hidden: 마지막 프롬프트 토큰, 1~32층 concat
    out = model(**enc, output_hidden_states=True, use_cache=True)
    last = L - 1
    hid = np.concatenate([out.hidden_states[k][0, last, :].float().cpu().numpy()
                          for k in range(1, len(out.hidden_states))]).astype(np.float32)
    # 2) 생성: 같은 kv-cache 이어서
    gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    ans = tok.decode(gen[0, L:], skip_special_tokens=True).strip()
    return hid, ans

HID = np.zeros((len(pairs), 32*4096), dtype=np.float32)
answers = []
for i, r in tqdm(pairs.iterrows(), total=len(pairs), desc="gen+hidden"):
    hid, ans = gen_and_hidden(r)
    HID[i] = hid
    answers.append(ans)
    if i % 100 == 0:
        gc.collect(); torch.cuda.empty_cache()

pairs = pairs.copy()
pairs["answer"] = answers
print("추출 완료:", HID.shape)
print(pairs[["kside","ctx_label","answer"]].head())

gen+hidden:   0%|          | 0/1224 [00:00<?, ?it/s]

추출 완료: (1224, 131072)
  kside ctx_label                                             answer
0    K+        C+                                             Action
1    K+        C+  The authors of "Free to Choose: A Personal Sta...
2    K+        C+                                The Czech Republic.
3    K+        C+  The text doesn't explicitly state that Barcelo...
4    K+        C+              The capital of Loir-et-Cher is Blois.


In [ ]:
# ── 1) hidden 즉시 백업 ──
np.save(f"{OUT_DIR}/hid_p8s.npy", HID)
np.save(f"{OUT_DIR}/hid_p8s_ids.npy", pairs["question"].values)
import shutil
for f in ["hid_p8s.npy", "hid_p8s_ids.npy"]:
    shutil.copy(f"{OUT_DIR}/{f}", f"{SAVE_DIR}/{f}")
print("hidden 백업 완료:", f"{SAVE_DIR}/hid_p8s.npy")

# ── 2) behavior 라벨링 ──
def label_behavior(row):
    ans = row.answer
    golds = json.loads(row.gold) if str(row.gold).startswith("[") else [row.gold]
    followed = match_gold(ans, [row.answer_surface])              # 문맥답 따라감
    # A_cb(closed-book 원답) 정규화 매칭 = parametric 유지
    para = match_gold(ans, [row.A_cb]) or match_gold(ans, golds if row.kside=="K+" else [row.A_cb])
    refused = bool(REFUSAL_PAT.search(ans))
    if refused:            return "abstain"
    if followed:           return "accept"
    if para:               return "retain"
    return "other"

pairs["behavior"] = pairs.apply(label_behavior, axis=1)
pairs["final_correct"] = pairs.apply(
    lambda r: match_gold(r.answer, json.loads(r.gold) if str(r.gold).startswith("[") else [r.gold]),
    axis=1)

print("=== behavior 분포 ===")
print(pairs.groupby(["kside","ctx_label"]).behavior.value_counts().unstack(fill_value=0))
print("\n=== K+/C− 셀 (w_main 추출 대상) ===")
kpcm = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")]
print(kpcm.behavior.value_counts().to_dict())

hidden 백업 완료: /content/drive/MyDrive/Grad_share/data_share/P9/hid_p8s.npy
=== behavior 분포 ===
behavior         abstain  accept  other  retain
kside ctx_label                                
K+    C+               0     288     30      60
      C-               1     224     48     105
K−    C+               0     180     52       2
      C-               0     188     44       2

=== K+/C− 셀 (w_main 추출 대상) ===
{'accept': 224, 'retain': 105, 'other': 48, 'abstain': 1}


In [ ]:
# ── other 스팟체크 ──
print("=== K+/C− other 예시 10개 ===")
for a in kpcm[kpcm.behavior=="other"].answer.head(10):
    print(" •", repr(a[:70]))

# ── layer 슬라이스 ──
def layer(H, L):  # L: 1~32 → concat 인덱스 L-1
    return H[:, (L-1)*4096:L*4096]

def diff_of_means(H, mask_a, mask_b):
    """레이어별 정규화된 방향. 반환: (32, 4096)"""
    W = np.zeros((32, 4096), dtype=np.float32)
    for L in range(1, 33):
        HL = layer(H, L)
        d = HL[mask_a].mean(0) - HL[mask_b].mean(0)
        W[L-1] = d / (np.linalg.norm(d) + 1e-8)
    return W

kpcm_idx = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")].index.values
m_acc = (pairs.loc[kpcm_idx,"behavior"]=="accept").values
m_ret = (pairs.loc[kpcm_idx,"behavior"]=="retain").values
Hk = HID[kpcm_idx]

w_main = diff_of_means(Hk, m_acc, m_ret)                          # 주입용

# w_pooled: 전체 accept vs retain
allm = pairs.behavior.isin(["accept","retain"]).values
w_pooled = diff_of_means(HID[allm],
                         (pairs[allm].behavior=="accept").values,
                         (pairs[allm].behavior=="retain").values)

# cos 비교 (레이어별)
def cos_layers(A, B):
    return np.array([np.dot(A[l],B[l])/((np.linalg.norm(A[l])*np.linalg.norm(B[l]))+1e-8)
                     for l in range(32)])

cos_mp = cos_layers(w_main, w_pooled)
print("\n=== cos(w_main, w_pooled) 레이어별 ===")
for L in [8,10,12,14,15,16,18,20]:
    print(f" L{L}: {cos_mp[L-1]:.3f}")

np.savez(f"{SAVE_DIR}/w_directions.npz", w_main=w_main, w_pooled=w_pooled)
print("\nw 저장 완료")

=== K+/C− other 예시 10개 ===
 • 'Action'
 • 'The authors of "Free to Choose: A Personal Statement" are Milton and R'
 • 'Unfortunately, the text does not mention the city where Offer Nissim w'
 • 'The text does not explicitly mention the father of the Chimera. Howeve'
 • 'Based on the text, Martin Garrix is a music producer and DJ, and his s'
 • 'The question is not asking about the author of "The Prince", but rathe'
 • 'According to the text, Ministry has been classified under many genres,'
 • 'The detective genre.'
 • 'The text does not explicitly state the genre of Frifot, but based on t'
 • 'The medieval Serbian state of Raška.'

=== cos(w_main, w_pooled) 레이어별 ===
 L8: 0.819
 L10: 0.836
 L12: 0.867
 L14: 0.909
 L15: 0.893
 L16: 0.873
 L18: 0.881
 L20: 0.875

w 저장 완료


In [ ]:
def label_behavior_v2(row):
    ans = row.answer
    golds = json.loads(row.gold) if str(row.gold).startswith("[") else [row.gold]
    if bool(REFUSAL_PAT.search(ans)) or "does not mention" in ans.lower() \
       or "does not explicitly" in ans.lower() or "doesn't mention" in ans.lower():
        return "abstain"
    followed = match_gold(ans, [row.answer_surface])          # 문맥 오답(치환값)
    para = match_gold(ans, [row.A_cb]) or match_gold(ans, golds)  # closed-book답 or gold
    if followed and not para: return "accept"
    if para and not followed: return "retain"
    if followed and para:     return "other"   # 둘 다 언급 = 모호
    return "other"

pairs["behavior"] = pairs.apply(label_behavior_v2, axis=1)
kpcm = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")]
print("K+/C− 재라벨:", kpcm.behavior.value_counts().to_dict())
print("\n전체 other 비율:", (pairs.behavior=="other").mean().round(3))
print("\n남은 other 예시:")
for a in kpcm[kpcm.behavior=="other"].answer.head(8): print(" •", repr(a[:70]))

K+/C− 재라벨: {'accept': 132, 'abstain': 91, 'other': 79, 'retain': 76}

전체 other 비율: 0.479

남은 other 예시:
 • 'Action'
 • 'The authors of "Free to Choose: A Personal Statement" are Milton and R'
 • 'The answer is Lincoln. The text actually describes the city of Lincoln'
 • 'The question is a bit misleading, as it suggests that Sally Mann is th'
 • 'Based on the text, Martin Garrix is a music producer and DJ, and his s'
 • 'The question is not asking about the author of "The Prince", but rathe'
 • 'According to the text, Ministry has been classified under many genres,'
 • 'The detective genre.'


In [ ]:
def label_behavior_v3(row):
    ans = row.answer
    golds = json.loads(row.gold) if str(row.gold).startswith("[") else [row.gold]
    followed = match_gold(ans, [row.answer_surface])   # 문맥이 주장한 치환 오답
    para     = match_gold(ans, golds)                  # 진짜 정답(=K+ parametric)
    refused  = bool(REFUSAL_PAT.search(ans))
    # 우선순위: 문맥답만 → accept / 정답만 → retain / 둘다 or 둘다아님 → 세부판정
    if followed and not para:  return "accept"
    if para and not followed:  return "retain"
    if refused:                return "abstain"
    if followed and para:      return "other"   # 진짜 모호(둘 다 명시)
    return "other"                              # 둘 다 아님(붕괴/무관)

pairs["behavior"] = pairs.apply(label_behavior_v3, axis=1)
kpcm = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")]
print("K+/C− 재라벨:", kpcm.behavior.value_counts().to_dict())
print("전체 other 비율:", (pairs.behavior=="other").mean().round(3))
print("\n남은 other 예시 8개:")
for a in kpcm[kpcm.behavior=="other"].answer.head(8): print(" •", repr(a[:75]))

K+/C− 재라벨: {'accept': 157, 'other': 116, 'retain': 105}
전체 other 비율: 0.568

남은 other 예시 8개:
 • 'Action'
 • 'The authors of "Free to Choose: A Personal Statement" are Milton and Rose D'
 • 'Unfortunately, the text does not mention the city where Offer Nissim was bo'
 • 'The text does not explicitly mention the father of the Chimera. However, it'
 • 'The text does not mention Vimmerby Municipality or its capital. It talks ab'
 • 'The answer is Lincoln. The text actually describes the city of Lincoln, not'
 • 'The question is a bit misleading, as it suggests that Sally Mann is the aut'
 • 'Based on the text, Martin Garrix is a music producer and DJ, and his signat'


In [ ]:
def label_behavior_v1(row):
    ans = row.answer
    golds = json.loads(row.gold) if str(row.gold).startswith("[") else [row.gold]
    followed = match_gold(ans, [row.answer_surface])
    para     = match_gold(ans, golds)
    if bool(REFUSAL_PAT.search(ans)): return "abstain"
    if followed: return "accept"       # followed 우선 (설명조 중복 언급 흡수)
    if para:     return "retain"
    return "other"

pairs["behavior"] = pairs.apply(label_behavior_v1, axis=1)
kpcm = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")]
print("K+/C− :", kpcm.behavior.value_counts().to_dict())
print("전체 other 비율:", (pairs.behavior=="other").mean().round(3))

K+/C− : {'accept': 224, 'retain': 105, 'other': 48, 'abstain': 1}
전체 other 비율: 0.116


In [ ]:
kpcm_idx = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")].index.values
m_acc = (pairs.loc[kpcm_idx,"behavior"]=="accept").values
m_ret = (pairs.loc[kpcm_idx,"behavior"]=="retain").values
Hk = HID[kpcm_idx]

w_main   = diff_of_means(Hk, m_acc, m_ret)
allm     = pairs.behavior.isin(["accept","retain"]).values
w_pooled = diff_of_means(HID[allm],
                         (pairs[allm].behavior=="accept").values,
                         (pairs[allm].behavior=="retain").values)

np.savez(f"{SAVE_DIR}/w_directions.npz", w_main=w_main, w_pooled=w_pooled)
print("w_main 추출:", w_main.shape, "| accept", m_acc.sum(), "retain", m_ret.sum())

# behavior 라벨도 저장 (steering 평가·재현용)
pairs.to_csv(f"{SAVE_DIR}/p8s_behavior.csv", index=False, encoding="utf-8-sig")
print("behavior csv 저장 완료")

w_main 추출: (32, 4096) | accept 224 retain 105
behavior csv 저장 완료


In [ ]:
import torch.nn.functional as F

# ── 레이어별 σ (주입 스케일 단위) : K+/C− hidden의 층별 표준편차 ──
sigma = np.array([layer(Hk, L).std() for L in range(1,33)], dtype=np.float32)  # (32,)

w_t = torch.tensor(w_main, device=model.device, dtype=torch.bfloat16)          # (32,4096)
sig_t = torch.tensor(sigma, device=model.device, dtype=torch.bfloat16)

# ── forward hook: decoder layer L 출력 residual에 coef*σ_L*w_L 가산 ──
_hooks, _cfg = [], {"coef":0.0, "layers":[]}
def make_hook(Lidx):
    def hook(mod, inp, out):
        if _cfg["coef"]==0.0 or Lidx not in _cfg["layers"]: return out
        h = out[0] if isinstance(out, tuple) else out
        h = h + _cfg["coef"]*sig_t[Lidx]*w_t[Lidx].to(h.dtype)
        return (h,)+out[1:] if isinstance(out, tuple) else h
    return hook

def attach():
    for Lidx in range(32):
        _hooks.append(model.model.layers[Lidx].register_forward_hook(make_hook(Lidx)))
def detach():
    for h in _hooks: h.remove()
    _hooks.clear()

@torch.no_grad()
def gen_steer(row, coef, layers):
    _cfg["coef"], _cfg["layers"] = coef, layers
    full = build_prompt_str(row)
    enc = tok(full, return_tensors="pt", add_special_tokens=False).to(model.device)
    L = enc["input_ids"].shape[1]
    gen = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(gen[0,L:], skip_special_tokens=True).strip()

# ── 평가셀 = K+/C− 전체 ──
eval_df = pairs.loc[kpcm_idx].reset_index(drop=True)
COEFS = [-2,-1.5,-1,-0.5,0,0.5,1,1.5,2,3]
BLOCKS = {"front":[8,9,10,11,12], "signal":[14,15,16,17,18]}

def em(ans, gold): return match_gold(ans, json.loads(gold) if str(gold).startswith("[") else [gold])
def foll(ans, surf): return match_gold(ans, [surf])
def collapsed(ans): return len(set(ans.split()))<=2 or len(ans)<3

attach()
rows=[]
for bname, blk in BLOCKS.items():
    for coef in tqdm(COEFS, desc=bname):
        e=f=c=0
        for _,r in eval_df.iterrows():
            a = gen_steer(r, coef, blk)
            e += em(a, r.gold); f += foll(a, r.answer_surface); c += collapsed(a)
        n=len(eval_df)
        rows.append(dict(block=bname, coef=coef, EM=e/n, followed=f/n, collapse=c/n))
detach()

sweep = pd.DataFrame(rows)
sweep.to_csv(f"{SAVE_DIR}/steer_sweep.csv", index=False)
print(sweep.to_string(index=False))

front:   0%|          | 0/10 [00:00<?, ?it/s]

signal:   0%|          | 0/10 [00:00<?, ?it/s]

 block  coef       EM  followed  collapse
 front  -2.0 0.484127  0.563492  0.201058
 front  -1.5 0.462963  0.579365  0.195767
 front  -1.0 0.468254  0.574074  0.187831
 front  -0.5 0.460317  0.595238  0.179894
 front   0.0 0.457672  0.595238  0.177249
 front   0.5 0.452381  0.619048  0.171958
 front   1.0 0.433862  0.621693  0.164021
 front   1.5 0.444444  0.632275  0.148148
 front   2.0 0.436508  0.624339  0.150794
 front   3.0 0.436508  0.642857  0.132275
signal  -2.0 0.500000  0.563492  0.145503
signal  -1.5 0.465608  0.584656  0.150794
signal  -1.0 0.476190  0.587302  0.161376
signal  -0.5 0.476190  0.597884  0.174603
signal   0.0 0.457672  0.595238  0.177249
signal   0.5 0.444444  0.605820  0.177249
signal   1.0 0.436508  0.626984  0.190476
signal   1.5 0.412698  0.634921  0.195767
signal   2.0 0.415344  0.645503  0.193122
signal   3.0 0.370370  0.656085  0.232804


In [ ]:
print(pairs.kside.unique())        # 실제 값 확인
print(pairs.ctx_label.unique())

['K+' 'K−']
['C+' 'C-']


In [ ]:
kminus = "K−"
eval_kmcp = pairs[(pairs.kside==kminus)&(pairs.ctx_label=="C+")].reset_index(drop=True)
print("n =", len(eval_kmcp))

MAX_NEW_FAST = 16
attach()
rows=[]; detail={}     # detail[coef] = 샘플별 정오 배열
for coef in tqdm([0, 2, 3], desc="K-/C+ accept"):
    ems=[]; folls=[]; cols=[]
    for _,r in eval_kmcp.iterrows():
        _cfg["coef"], _cfg["layers"] = coef, [14,15,16,17,18]
        full = build_prompt_str(r)
        enc = tok(full, return_tensors="pt", add_special_tokens=False).to(model.device)
        L = enc["input_ids"].shape[1]
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_FAST, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        a = tok.decode(gen[0,L:], skip_special_tokens=True).strip()
        ems.append(em(a, r.gold)); folls.append(foll(a, r.answer_surface)); cols.append(collapsed(a))
    detail[coef] = np.array(ems, dtype=int)
    rows.append(dict(coef=coef, EM=np.mean(ems), followed=np.mean(folls), collapse=np.mean(cols)))
detach()

acc_sweep = pd.DataFrame(rows); print(acc_sweep.to_string(index=False))

# ── 회복률/손상률 (baseline=coef0 대비) ──
base = detail[0]
for coef in [2,3]:
    cur = detail[coef]
    recovered = ((base==0)&(cur==1)).sum()   # 틀렸다 → 맞음
    broken    = ((base==1)&(cur==0)).sum()   # 맞았다 → 틀림
    print(f"coef {coef}: 회복 {recovered} (틀린 것 중 {recovered/max((base==0).sum(),1):.1%}), "
          f"손상 {broken} (맞은 것 중 {broken/max((base==1).sum(),1):.1%}), 순증 {recovered-broken}")

np.savez(f"{SAVE_DIR}/acc_detail.npz", **{f"c{k}":v for k,v in detail.items()})

n = 234


K-/C+ accept:   0%|          | 0/3 [00:00<?, ?it/s]

 coef       EM  followed  collapse
    0 0.752137  0.662393  0.465812
    2 0.794872  0.705128  0.363248
    3 0.816239  0.722222  0.329060
coef 2: 회복 13 (틀린 것 중 22.4%), 손상 3 (맞은 것 중 1.7%), 순증 10
coef 3: 회복 17 (틀린 것 중 29.3%), 손상 2 (맞은 것 중 1.1%), 순증 15


In [ ]:
eval_kpcm = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")].reset_index(drop=True)
attach()
res={}
for bname, blk in {"front":[8,9,10,11,12],"signal":[14,15,16,17,18]}.items():
    for coef in tqdm([0,-2,-3], desc=f"retain-{bname}"):
        ems=[]
        for _,r in eval_kpcm.iterrows():
            _cfg["coef"], _cfg["layers"]=coef, blk
            full=build_prompt_str(r)
            enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
            L=enc["input_ids"].shape[1]
            gen=model.generate(**enc,max_new_tokens=16,do_sample=False,pad_token_id=tok.pad_token_id)
            ems.append(em(tok.decode(gen[0,L:],skip_special_tokens=True).strip(), r.gold))
        res[(bname,coef)]=np.array(ems,dtype=int)
detach()

for bname in ["front","signal"]:
    base=res[(bname,0)]
    print(f"\n[{bname}] baseline EM {base.mean():.3f}")
    for coef in [-2,-3]:
        cur=res[(bname,coef)]
        rec=((base==0)&(cur==1)).sum(); brk=((base==1)&(cur==0)).sum()
        print(f"  coef{coef}: EM {cur.mean():.3f} | 회복 {rec} ({rec/max((base==0).sum(),1):.1%}) "
              f"손상 {brk} ({brk/max((base==1).sum(),1):.1%}) 순증 {rec-brk}")
np.savez(f"{SAVE_DIR}/retain_detail.npz", **{f"{b}_{c}":v for (b,c),v in res.items()})

retain-front:   0%|          | 0/3 [00:00<?, ?it/s]

retain-signal:   0%|          | 0/3 [00:00<?, ?it/s]


[front] baseline EM 0.243
  coef-2: EM 0.262 | 회복 10 (3.5%) 손상 3 (3.3%) 순증 7
  coef-3: EM 0.265 | 회복 14 (4.9%) 손상 6 (6.5%) 순증 8

[signal] baseline EM 0.243
  coef-2: EM 0.235 | 회복 10 (3.5%) 손상 13 (14.1%) 순증 -3
  coef-3: EM 0.235 | 회복 13 (4.5%) 손상 16 (17.4%) 순증 -3


In [ ]:
eval_kpcm = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")].reset_index(drop=True)

attach()
res48={}
for bname, blk in {"front":[8,9,10,11,12],"signal":[14,15,16,17,18]}.items():
    for coef in tqdm([0,-2,-3], desc=f"retain48-{bname}"):
        ems=[]
        for _,r in eval_kpcm.iterrows():
            _cfg["coef"], _cfg["layers"]=coef, blk
            full=build_prompt_str(r)
            enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
            L=enc["input_ids"].shape[1]
            gen=model.generate(**enc,max_new_tokens=48,do_sample=False,pad_token_id=tok.pad_token_id)
            ems.append(em(tok.decode(gen[0,L:],skip_special_tokens=True).strip(), r.gold))
        res48[(bname,coef)]=np.array(ems,dtype=int)
detach()

for bname in ["front","signal"]:
    base=res48[(bname,0)]
    print(f"\n[{bname}] baseline EM {base.mean():.3f}  (48토큰, CELL7의 0.458과 대조)")
    for coef in [-2,-3]:
        cur=res48[(bname,coef)]
        rec=((base==0)&(cur==1)).sum(); brk=((base==1)&(cur==0)).sum()
        print(f"  coef{coef}: EM {cur.mean():.3f} | 회복 {rec} ({rec/max((base==0).sum(),1):.1%}) "
              f"손상 {brk} ({brk/max((base==1).sum(),1):.1%}) 순증 {rec-brk}")
np.savez(f"{SAVE_DIR}/retain_detail48.npz", **{f"{b}_{c}":v for (b,c),v in res48.items()})

retain48-front:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# 랜덤 방향: w_main과 동일하게 레이어별 단위벡터 (σ 스케일은 훅에서 동일 적용)
def random_dirs(seed):
    rng = np.random.default_rng(seed)
    W = rng.standard_normal((32,4096)).astype(np.float32)
    return W / (np.linalg.norm(W,axis=1,keepdims=True)+1e-8)

eval_kmcp = pairs[(pairs.kside=="K−")&(pairs.ctx_label=="C+")].reset_index(drop=True)
base_acc = np.load(f"{SAVE_DIR}/acc_detail.npz")["c0"]   # accept baseline 개별기록 재사용

SEEDS = [0,1,2]
print(f"[참조] 실제 w_main coef+3: 회복 17 / 손상 2 (틀린것 29.3%)\n")
for sd in SEEDS:
    Wr = torch.tensor(random_dirs(sd), device=model.device, dtype=torch.bfloat16)
    w_backup = w_t.clone(); w_t.copy_(Wr)          # 훅이 참조하는 w_t를 랜덤으로 교체
    attach(); ems=[]
    for _,r in eval_kmcp.iterrows():
        _cfg["coef"], _cfg["layers"] = 3, [14,15,16,17,18]
        full=build_prompt_str(r)
        enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
        L=enc["input_ids"].shape[1]
        gen=model.generate(**enc,max_new_tokens=16,do_sample=False,pad_token_id=tok.pad_token_id)
        ems.append(em(tok.decode(gen[0,L:],skip_special_tokens=True).strip(), r.gold))
    detach(); w_t.copy_(w_backup)                  # 원복
    cur=np.array(ems,dtype=int)
    rec=((base_acc==0)&(cur==1)).sum(); brk=((base_acc==1)&(cur==0)).sum()
    print(f"seed{sd}: EM {cur.mean():.3f} | 회복 {rec} ({rec/max((base_acc==0).sum(),1):.1%}) "
          f"손상 {brk} ({brk/max((base_acc==1).sum(),1):.1%}) 순증 {rec-brk}")

[참조] 실제 w_main coef+3: 회복 17 / 손상 2 (틀린것 29.3%)

seed0: EM 0.752 | 회복 0 (0.0%) 손상 0 (0.0%) 순증 0
seed1: EM 0.748 | 회복 0 (0.0%) 손상 1 (0.6%) 순증 -1
seed2: EM 0.756 | 회복 2 (3.4%) 손상 1 (0.6%) 순증 1


In [ ]:
print("w_t 원복 확인:", torch.allclose(w_t.cpu().float(), torch.tensor(w_main)))  # True여야

w_t 원복 확인: False


In [ ]:
W = np.load(f"{SAVE_DIR}/w_directions.npz")
w_main = W["w_main"]
w_t = torch.tensor(w_main, device=model.device, dtype=torch.bfloat16)
print("복구 확인:", torch.allclose(w_t.cpu().float(), torch.tensor(w_main)))  # True

복구 확인: False


In [ ]:
W = np.load(f"{SAVE_DIR}/w_directions.npz")
w_main = W["w_main"].astype(np.float32)
w_t = torch.tensor(w_main, device=model.device, dtype=torch.bfloat16)

# bf16 왕복 기준으로 비교 (이게 맞는 비교)
ref = torch.tensor(w_main).to(torch.bfloat16).float()
print("bf16 기준 일치:", torch.allclose(w_t.cpu().float(), ref, atol=1e-2))

# 랜덤과 확실히 다른지: L15 벡터 앞 5개 값 눈으로
print("w_t[15][:5] :", w_t[15][:5].cpu().float().numpy())
print("w_main[15][:5]:", w_main[15][:5])

bf16 기준 일치: True
w_t[15][:5] : [ 7.4863434e-05  7.3242188e-03 -1.4465332e-02  4.8522949e-03
  1.2817383e-02]
w_main[15][:5]: [ 7.48750754e-05  7.32825557e-03 -1.44556882e-02  4.84662177e-03
  1.28214825e-02]


In [ ]:
attach()
ems=[]
for _,r in eval_kmcp.head(50).iterrows():
    _cfg["coef"], _cfg["layers"] = 3, [14,15,16,17,18]
    full=build_prompt_str(r)
    enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
    L=enc["input_ids"].shape[1]
    gen=model.generate(**enc,max_new_tokens=16,do_sample=False,pad_token_id=tok.pad_token_id)
    ems.append(em(tok.decode(gen[0,L:],skip_special_tokens=True).strip(), r.gold))
print("50샘플 EM:", np.mean(ems))
detach()

50샘플 EM: 0.82


In [ ]:
def cluster_bootstrap(base, cur, groups, n_boot=2000, seed=0):
    """질문 단위 리샘플로 회복/손상/순증 CI. base,cur: 0/1 배열, groups: 질문id 배열"""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    idx_by_q = {q: np.where(groups==q)[0] for q in uniq}
    recs, brks, nets = [], [], []
    for _ in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        rows = np.concatenate([idx_by_q[q] for q in pick])
        b, c = base[rows], cur[rows]
        rec = ((b==0)&(c==1)).sum(); brk = ((b==1)&(c==0)).sum()
        recs.append(rec); brks.append(brk); nets.append(rec-brk)
    f = lambda a: (np.percentile(a,2.5), np.percentile(a,97.5))
    return dict(rec_CI=f(recs), brk_CI=f(brks), net_CI=f(nets),
                net_mean=np.mean(nets))

# ── accept: K−/C+ signal +3 ──
acc = np.load(f"{SAVE_DIR}/acc_detail.npz")
g_acc = eval_kmcp["question"].values
r = cluster_bootstrap(acc["c0"], acc["c3"], g_acc)
print(f"[accept K−/C+ signal+3] 순증 {r['net_mean']:.1f}  CI{r['net_CI']}  "
      f"회복CI{r['rec_CI']} 손상CI{r['brk_CI']}")

# ── retain: K+/C− front/signal −3 (16토큰) ──
ret = np.load(f"{SAVE_DIR}/retain_detail.npz")
g_ret = eval_kpcm["question"].values
for bname in ["front","signal"]:
    r = cluster_bootstrap(ret[f"{bname}_0"], ret[f"{bname}_-3"], g_ret)
    sig = "유의(+)" if r['net_CI'][0]>0 else ("유의(−)" if r['net_CI'][1]<0 else "불명확")
    print(f"[retain {bname} −3] 순증 {r['net_mean']:.1f}  CI{r['net_CI']}  → {sig}")

[accept K−/C+ signal+3] 순증 15.1  CI(np.float64(7.0), np.float64(24.0))  회복CI(np.float64(10.0), np.float64(26.0)) 손상CI(np.float64(0.0), np.float64(5.0))
[retain front −3] 순증 8.0  CI(np.float64(0.0), np.float64(17.0))  → 불명확
[retain signal −3] 순증 -3.1  CI(np.float64(-14.0), np.float64(7.0))  → 불명확


In [ ]:
ret_ext = {}

In [ ]:
attach()
for bname, blk in {"front":[8,9,10,11,12],"signal":[14,15,16,17,18]}.items():
    for coef in tqdm([-4,-5], desc=f"retain-ext-{bname}"):
        ems=[]
        for _,r in eval_kpcm.iterrows():
            _cfg["coef"], _cfg["layers"]=coef, blk
            full=build_prompt_str(r)
            enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
            L=enc["input_ids"].shape[1]
            gen=model.generate(**enc,max_new_tokens=16,do_sample=False,pad_token_id=tok.pad_token_id)
            ems.append(em(tok.decode(gen[0,L:],skip_special_tokens=True).strip(), r.gold))
        ret_ext[(bname,coef)]=np.array(ems,dtype=int)   # 아래서 dict 먼저 생성
detach()

retain-ext-front:   0%|          | 0/2 [00:00<?, ?it/s]

retain-ext-signal:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
retain16 = np.load(f"{SAVE_DIR}/retain_detail.npz")
g_ret = eval_kpcm["question"].values
for bname in ["front","signal"]:
    base = retain16[f"{bname}_0"]
    print(f"\n[{bname}] baseline EM {base.mean():.3f}")
    for coef in [-4,-5]:
        cur = ret_ext[(bname,coef)]
        rec=((base==0)&(cur==1)).sum(); brk=((base==1)&(cur==0)).sum(); col=collapsed
        r = cluster_bootstrap(base, cur, g_ret)
        print(f"  coef{coef}: EM {cur.mean():.3f} 회복 {rec} 손상 {brk} 순증 {rec-brk} "
              f"CI{r['net_CI']}")
np.savez(f"{SAVE_DIR}/retain_ext16.npz", **{f"{b}_{c}":v for (b,c),v in ret_ext.items()})


[front] baseline EM 0.243
  coef-4: EM 0.251 회복 14 손상 11 순증 3 CI(np.float64(-7.0), np.float64(13.0))
  coef-5: EM 0.267 회복 23 손상 14 순증 9 CI(np.float64(-2.0), np.float64(21.0))

[signal] baseline EM 0.243
  coef-4: EM 0.222 회복 18 손상 26 순증 -8 CI(np.float64(-21.0), np.float64(5.0))
  coef-5: EM 0.198 회복 18 손상 35 순증 -17 CI(np.float64(-31.0), np.float64(-3.0))


In [ ]:
ret48 = {}
attach()
for bname, blk in {"front":[8,9,10,11,12],"signal":[14,15,16,17,18]}.items():
    for coef in tqdm([0,-2,-3,-4,-5], desc=f"retain48-{bname}"):
        ems=[]; cols=[]
        for _,r in eval_kpcm.iterrows():
            _cfg["coef"], _cfg["layers"]=coef, blk
            full=build_prompt_str(r)
            enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
            L=enc["input_ids"].shape[1]
            gen=model.generate(**enc,max_new_tokens=48,do_sample=False,pad_token_id=tok.pad_token_id)
            a=tok.decode(gen[0,L:],skip_special_tokens=True).strip()
            ems.append(em(a,r.gold)); cols.append(collapsed(a))
        ret48[(bname,coef)]=(np.array(ems,dtype=int), np.mean(cols))
detach()

g_ret = eval_kpcm["question"].values
for bname in ["front","signal"]:
    base = ret48[(bname,0)][0]
    print(f"\n[{bname}] baseline EM {base.mean():.3f} (48토큰)")
    for coef in [-2,-3,-4,-5]:
        cur,col = ret48[(bname,coef)]
        rec=((base==0)&(cur==1)).sum(); brk=((base==1)&(cur==0)).sum()
        r = cluster_bootstrap(base, cur, g_ret)
        sig = "유의(+)" if r['net_CI'][0]>0 else ("유의(−)" if r['net_CI'][1]<0 else "불명확")
        print(f"  coef{coef}: EM {cur.mean():.3f} collapse {col:.2f} "
              f"순증 {rec-brk} CI{r['net_CI']} → {sig}")
np.savez(f"{SAVE_DIR}/retain48.npz", **{f"{b}_{c}":v[0] for (b,c),v in ret48.items()})

retain48-front:   0%|          | 0/5 [00:00<?, ?it/s]

retain48-signal:   0%|          | 0/5 [00:00<?, ?it/s]


[front] baseline EM 0.458 (48토큰)
  coef-2: EM 0.484 collapse 0.20 순증 10 CI(np.float64(0.0), np.float64(20.0)) → 불명확
  coef-3: EM 0.474 collapse 0.23 순증 6 CI(np.float64(-7.0), np.float64(19.0)) → 불명확
  coef-4: EM 0.463 collapse 0.26 순증 2 CI(np.float64(-13.0), np.float64(18.0)) → 불명확
  coef-5: EM 0.471 collapse 0.29 순증 5 CI(np.float64(-9.0), np.float64(21.0)) → 불명확

[signal] baseline EM 0.458 (48토큰)
  coef-2: EM 0.500 collapse 0.15 순증 16 CI(np.float64(2.0), np.float64(30.0)) → 유의(+)
  coef-3: EM 0.511 collapse 0.12 순증 20 CI(np.float64(4.0), np.float64(36.0)) → 유의(+)
  coef-4: EM 0.542 collapse 0.09 순증 32 CI(np.float64(14.0), np.float64(51.0)) → 유의(+)
  coef-5: EM 0.542 collapse 0.07 순증 32 CI(np.float64(13.0), np.float64(51.0)) → 유의(+)


In [ ]:
# w_main 정상인지 먼저 확인 (랜덤 오염 방지)
W = np.load(f"{SAVE_DIR}/w_directions.npz"); w_main = W["w_main"].astype(np.float32)
w_t = torch.tensor(w_main, device=model.device, dtype=torch.bfloat16)

base_ret48 = np.load(f"{SAVE_DIR}/retain48.npz")["signal_0"]   # 48토큰 signal baseline
g_ret = eval_kpcm["question"].values

print(f"[참조] 실제 w_main signal−4: 순증 32, CI[14,51]\n")
for sd in [0,1,2]:
    rng = np.random.default_rng(sd)
    Wr = rng.standard_normal((32,4096)).astype(np.float32)
    Wr = Wr/(np.linalg.norm(Wr,axis=1,keepdims=True)+1e-8)
    w_backup = w_t.clone(); w_t.copy_(torch.tensor(Wr,device=model.device,dtype=torch.bfloat16))
    attach(); ems=[]
    for _,r in eval_kpcm.iterrows():
        _cfg["coef"], _cfg["layers"] = -4, [14,15,16,17,18]
        full=build_prompt_str(r)
        enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
        L=enc["input_ids"].shape[1]
        gen=model.generate(**enc,max_new_tokens=48,do_sample=False,pad_token_id=tok.pad_token_id)
        ems.append(em(tok.decode(gen[0,L:],skip_special_tokens=True).strip(), r.gold))
    detach(); w_t.copy_(w_backup)
    cur=np.array(ems,dtype=int)
    rec=((base_ret48==0)&(cur==1)).sum(); brk=((base_ret48==1)&(cur==0)).sum()
    rb = cluster_bootstrap(base_ret48, cur, g_ret)
    print(f"seed{sd}: EM {cur.mean():.3f} 회복 {rec} 손상 {brk} 순증 {rec-brk} CI{rb['net_CI']}")

# 원복 확실히
w_t.copy_(torch.tensor(w_main,device=model.device,dtype=torch.bfloat16))
print("\nw_t 복구 완료 (bf16 기준)")

[참조] 실제 w_main signal−4: 순증 32, CI[14,51]

seed0: EM 0.481 회복 21 손상 12 순증 9 CI(np.float64(-2.0), np.float64(20.0))
seed1: EM 0.460 회복 12 손상 11 순증 1 CI(np.float64(-8.0), np.float64(11.0))
seed2: EM 0.455 회복 11 손상 12 순증 -1 CI(np.float64(-10.0), np.float64(8.0))

w_t 복구 완료 (bf16 기준)


In [ ]:
eval_kmcp = pairs[(pairs.kside=="K−")&(pairs.ctx_label=="C+")].reset_index(drop=True)
acc = np.load(f"{SAVE_DIR}/acc_detail.npz")   # signal 결과(c0,c2,c3) 재사용
base_acc = acc["c0"]
g_acc = eval_kmcp["question"].values

acc_front = {}
attach()
for coef in tqdm([2,3], desc="accept-front"):
    ems=[]
    for _,r in eval_kmcp.iterrows():
        _cfg["coef"], _cfg["layers"] = coef, [8,9,10,11,12]
        full=build_prompt_str(r)
        enc=tok(full,return_tensors="pt",add_special_tokens=False).to(model.device)
        L=enc["input_ids"].shape[1]
        gen=model.generate(**enc,max_new_tokens=16,do_sample=False,pad_token_id=tok.pad_token_id)
        ems.append(em(tok.decode(gen[0,L:],skip_special_tokens=True).strip(), r.gold))
    acc_front[coef]=np.array(ems,dtype=int)
detach()

print(f"baseline EM {base_acc.mean():.3f}")
print("[signal 참조] +3: 순증 15, CI[7,24]\n")
for coef in [2,3]:
    cur=acc_front[coef]
    rec=((base_acc==0)&(cur==1)).sum(); brk=((base_acc==1)&(cur==0)).sum()
    r=cluster_bootstrap(base_acc, cur, g_acc)
    sig="유의(+)" if r['net_CI'][0]>0 else "불명확"
    print(f"front +{coef}: EM {cur.mean():.3f} 회복 {rec} 손상 {brk} 순증 {rec-brk} CI{r['net_CI']} → {sig}")
np.savez(f"{SAVE_DIR}/acc_front.npz", **{f"c{k}":v for k,v in acc_front.items()})

accept-front:   0%|          | 0/2 [00:00<?, ?it/s]

baseline EM 0.752
[signal 참조] +3: 순증 15, CI[7,24]

front +2: EM 0.748 회복 2 손상 3 순증 -1 CI(np.float64(-5.0), np.float64(3.0)) → 불명확
front +3: EM 0.744 회복 2 손상 4 순증 -2 CI(np.float64(-7.0), np.float64(3.0)) → 불명확
